In [1]:
import numpy as np
import pandas as pd
import matplotlib
import plotly
import os
import glob
import plotly.express as px

In [2]:
#path to the experiments log folder
log_data_dir = "/nfs/lslab2/fireline/simharness/experiments/"

#model type in experiment folder
model = 'DQN'

#output folder
outputs = "exp_reports/"

#enter the experiment number here
exp = '2023-09-28_21-12-58'

In [3]:
#concatenate the experiment directory path
exp_dir = log_data_dir + exp + "/" + model + "/"

#get the name of the directory that holds the gifs and csv logs
log_dir_name = [s for s in os.listdir(exp_dir) if model in s][0]

#get the full path of the log directory 
log_dir = exp_dir + log_dir_name + "/"

print("Directory with callback logs and gifs:")
print(log_dir)
print("Directory Contains:")
print(os.listdir(log_dir))



Directory with callback logs and gifs:
/nfs/lslab2/fireline/simharness/experiments/2023-09-28_20-06-10/DQN/DQN_simharness2.environments.reactive.ReactiveHarness_7d888_00000_0_2023-09-28_20-06-22/
Directory Contains:
['params.json', 'params.pkl', 'result.json', 'events.out.tfevents.1695931698.sh2DG', 'progress.csv', 'gifs', 'sim_data', 'benchsim_data', 'agent_data']


### Visualize the Benchsim Results

In [4]:
#get the most recent log in the benchsim folder for the bench_sim stats
##this assumes that all the benchsim stats are the same
benchsim_eval_log = max(([os.path.join((log_dir + "benchsim_data/"), basename) for basename in os.listdir((log_dir + "benchsim_data/"))]), key=os.path.getctime)
print(benchsim_eval_log)

#turn the benchsim data into a dataframe
benchsim_data = pd.read_csv(benchsim_eval_log, header=0, index_col=0)
print(benchsim_data.head())

/nfs/lslab2/fireline/simharness/experiments/2023-09-28_20-06-10/DQN/DQN_simharness2.environments.reactive.ReactiveHarness_7d888_00000_0_2023-09-28_20-06-22/benchsim_data/eval_iter_16.csv
          timestep  burned  burning  unburned
sim_step                                     
0                0       0        4     16380
1                4       0        4     16380
2                8       0        4     16380
3               12       0        4     16380
4               16       1        3     16380


### Visualize a Single Agent Results

#### Most Recent Agent Results

In [6]:
#get the most recent log in the sim folder for the agent_sim stats
#agent_eval_log = max(([os.path.join((log_dir + "sim_data/"), basename) for basename in os.listdir((log_dir + "sim_data/"))]), key=os.path.getctime)
#print(agent_eval_log)

#or choose an agent_eval_log
agent_iter_number = 1
agent_eval_log = log_dir + "sim_data/eval_iter_" + str(agent_iter_number) + ".csv"

#turn the benchsim data into a dataframe
agent_data = pd.read_csv(agent_eval_log, header=0, index_col=0)

In [7]:
#merge in the benchmark sim data to create the simharness_analytics dataframe
benchmark_analytics = benchsim_data[['timestep', 'burned', 'burning', 'unburned']].copy()
benchmark_analytics.rename(columns={"burned": "unmitigated_burned", "burning": "unmitigated_burning", "unburned": "unmitigated_unburned"}, inplace=True)
eval_analytics = agent_data.merge(benchmark_analytics, how='inner', on='timestep')
print(eval_analytics.head())

   timestep  burned  burning  unburned  mitigated  agent_interactions  \
0         0       0        3     16380          1                   1   
1         4       0        3     16380          1                   4   
2         8       0        4     16379          1                   4   
3        12       0        6     16377          1                   4   
4        16       0       11     16372          1                   4   

   agent_movements  unmitigated_burned  unmitigated_burning  \
0                1                   0                    4   
1                4                   0                    4   
2                4                   0                    4   
3                4                   0                    4   
4                4                   1                    3   

   unmitigated_unburned  
0                 16380  
1                 16380  
2                 16380  
3                 16380  
4                 16380  


In [8]:
#sh_burned_plot = px.line(eval_analytics, x="timestep", y=['mitigated','burned','unmitigated_burned'], title='SimHarness Performance Analysis')
#sh_burned_plot.show()

## Cumulative Agent Analysis

In [10]:
agent_log_dir = log_dir + "sim_data/"
simharness_analytics = pd.DataFrame(columns=['Eval_Num', 'Num_timesteps', 'Num_burned', 'Num_unburned', 'Num_mitigated'])

for basepath in os.listdir(agent_log_dir):
    #get the eval_num from the csv file name
    eval_num = basepath.replace('eval_iter_', '')
    eval_num = eval_num.replace('.csv', '')
    eval_num = int(eval_num)
    
    #take the last row of the eval_iter_#.csv and translate those values into our dataframe
    single_agent_data = pd.read_csv((agent_log_dir + basepath), header=0, index_col=0)
    last_row = single_agent_data.iloc[-1:]
    #print(type(last_row.timestep.iloc[0]))
    final_run_data= [eval_num, int((last_row.timestep.iloc[0])), int((last_row.burned.iloc[0])), int((last_row.unburned.iloc[0])),int((last_row.mitigated.iloc[0]))]
    
    #place that last row data into the dataframe
    simharness_analytics.loc[len(simharness_analytics)] = final_run_data

print(simharness_analytics.head())
    

   Eval_Num  Num_timesteps  Num_burned  Num_unburned  Num_mitigated
0         1             28           7         16349             28
1         2             40           8         16348             28
2         3            580        1157         15164             63
3         4             20           1         16367             16
4         5            124          91         16207             86


In [11]:
#merge these with the benchsim results
bench_log_dir = log_dir + "bench_data/"
bench_analytics = pd.DataFrame(columns=['Eval_Num', 'Bench_timesteps', 'Bench_burned', 'Bench_unburned'])

for basepath in os.listdir(agent_log_dir):
    #get the eval_num from the csv file name
    eval_num = basepath.replace('eval_iter_', '')
    eval_num = eval_num.replace('.csv', '')
    eval_num = int(eval_num)
    
    #take the last row of the eval_iter_#.csv and translate those values into our dataframe
    single_eval_data = pd.read_csv((agent_log_dir + basepath), header=0, index_col=0)
    last_row = single_eval_data.iloc[-1:]
    #print(type(last_row.timestep.iloc[0]))
    final_run_data= [eval_num, int((last_row.timestep.iloc[0])), int((last_row.burned.iloc[0])), int((last_row.unburned.iloc[0]))]
    
    #place that last row data into the dataframe
    bench_analytics.loc[len(bench_analytics)] = final_run_data
    
#merge with the simharness analytics
simharness_analytics = simharness_analytics.merge(bench_analytics, how='inner', on='Eval_Num')
print(simharness_analytics.head())

   Eval_Num  Num_timesteps  Num_burned  Num_unburned  Num_mitigated  \
0         1             28           7         16349             28   
1         2             40           8         16348             28   
2         3            580        1157         15164             63   
3         4             20           1         16367             16   
4         5            124          91         16207             86   

   Bench_timesteps  Bench_burned  Bench_unburned  
0               28             7           16349  
1               40             8           16348  
2              580          1157           15164  
3               20             1           16367  
4              124            91           16207  


## Add comparison metrics

In [14]:
simharness_analytics["Agent_burn_rate"] = simharness_analytics["Num_burned"]/simharness_analytics["Num_timesteps"]
simharness_analytics["Bench_burn_rate"] = simharness_analytics["Bench_burned"]/simharness_analytics["Bench_timesteps"]

simharness_analytics["Num_Saved"] = simharness_analytics["Num_unburned"] - simharness_analytics["Bench_unburned"]
simharness_analytics["Percent_Area_Saved"] = (simharness_analytics["Num_Saved"]/(simharness_analytics["Bench_burned"] + simharness_analytics["Bench_burned"]))*100
simharness_analytics["Time_Saved"] = simharness_analytics["Num_timesteps"] - simharness_analytics["Bench_timesteps"]
simharness_analytics["Percent_Time_Saved"] = (simharness_analytics["Time_Saved"]/(simharness_analytics["Bench_timesteps"]))*100

simharness_analytics["Percent_Burn_Rate_Reduced"] = 100 - (((simharness_analytics["Agent_burn_rate"])/(simharness_analytics["Bench_burn_rate"]))*100)
print(simharness_analytics.head())

   Eval_Num  Num_timesteps  Num_burned  Num_unburned  Num_mitigated  \
0         1             28           7         16349             28   
1         2             40           8         16348             28   
2         3            580        1157         15164             63   
3         4             20           1         16367             16   
4         5            124          91         16207             86   

   Bench_timesteps  Bench_burned  Bench_unburned  Agent_burn_rate  \
0               28             7           16349         0.250000   
1               40             8           16348         0.200000   
2              580          1157           15164         1.994828   
3               20             1           16367         0.050000   
4              124            91           16207         0.733871   

   Bench_burn_rate  Num_Saved  Percent_Area_Saved  Time_Saved  \
0         0.250000          0                 0.0           0   
1         0.200000          

In [17]:
agent_full_performance_plot = px.line(simharness_analytics, x="Eval_Num", y=["Percent_Area_Saved","Percent_Time_Saved","Percent_Burn_Rate_Reduced"], title='SimHarness Performance Analysis')
agent_full_performance_plot.show()

In [15]:
area_saved_plot = px.line(simharness_analytics, x="Eval_Num", y=["Num_Saved"], title='Area Saved by Agent')
area_saved_plot.show()

In [16]:
time_saved_plot = px.line(simharness_analytics, x="Eval_Num", y=["Time_Saved"], title='Time Saved by Agent')
time_saved_plot.show()